<a href="https://colab.research.google.com/github/parulgoell/ai-attendance-agent/blob/main/smart__bus_routing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import heapq   # Import heapq to use a priority queue for Dijkstra’s algorithm

# ---- City graph: each stop maps to its neighbors and travel time ----
graph = {
    'A': {'B': 4, 'C': 2},   # From A → B takes 4 mins, A → C takes 2 mins
    'B': {'D': 5},           # From B → D takes 5 mins
    'C': {'B': 1, 'D': 8},   # From C → B takes 1 min, C → D takes 8 mins
    'D': {}                  # D has no outgoing routes
}

# Passenger wait times before optimization
wait_time = {
    'A': 10,   # People at stop A typically wait 10 mins
    'B': 12,   # People at stop B wait 12 mins
    'C': 8,    # People at stop C wait 8 mins
    'D': 15    # People at stop D wait 15 mins
}

# ---- Dijkstra shortest path function ----
def shortest_paths(graph, start):

    # Create a dictionary with infinite time for every stop initially
    dist = {node: float('inf') for node in graph}

    # Distance to starting stop is 0 (bus begins here)
    dist[start] = 0

    # Priority queue storing (travel_time, stop)
    pq = [(0, start)]

    # Main loop: process stops with smallest travel time first
    while pq:
        d, node = heapq.heappop(pq)  # Get stop with smallest travel time so far

        # Skip if this is not the latest/valid shortest distance
        if d != dist[node]:
            continue

        # Explore all neighboring stops from the current stop
        for nxt, cost in graph[node].items():

            # New distance = current distance + edge cost
            nd = d + cost

            # Update only if we found a shorter way to reach 'nxt'
            if nd < dist[nxt]:
                dist[nxt] = nd                 # Update shortest time
                heapq.heappush(pq, (nd, nxt))  # Push new best distance into queue

    return dist  # Return all shortest travel times from the starting point


# Bus starts at stop A
travel_time = shortest_paths(graph, 'A')  # Compute fastest travel times to each stop

# ---- Compute how much passenger waiting time improves ----
reduction = {
    s: max(0, wait_time[s] - travel_time[s])  # New wait = old wait - bus arrival time
    for s in graph
}

# Add up all wait-time improvements for the entire bus network
total_reduction = sum(reduction.values())

# ---- Output results ----
print("Shortest travel time:", travel_time)        # Show optimal bus travel times
print("Wait-time reduction per stop:", reduction)  # Improvement at each stop
print("Total wait-time reduction:", total_reduction)  # Overall benefit


Shortest travel time: {'A': 0, 'B': 3, 'C': 2, 'D': 8}
Wait-time reduction per stop: {'A': 10, 'B': 9, 'C': 6, 'D': 7}
Total wait-time reduction: 32
